For this project, I am using run7722all_xyz.root that was produced by Arran using his custom Data processor that includes the muon hits (muon_paddle_hits) and the positions of each paddle in the meta tree.<br>

The data contains the data processed for the water PMTs also to link the muon hits with the PMT pulses.<br>


In [ ]:
# This block install some 3D plotting capability to your python install. 
# Uncomment and run once to install ipymp or plotly
#%pip install ipympl
#%pip install plotly

In [ ]:
# Let's print out the output tree.
import ROOT
f = ROOT.TFile.Open("../Data/run7722all_xyz.root")
output = f.Get("output")
output.Print()

The print out shows that the output tree has 353719 events. The useful variables for this analysis are
- muon_paddle_hits: 1s and 0s whether or not the paddle is hit out of the 134 total.
- digitNHits or digitNhitsCleaned: This is tell us how much energy is deposited in the water tank.
- digitCharge: Same as above but more direct correlation to energy.

In [ ]:
# This block just check if the digitNhits are valid for this dataset.
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
# Create RDataFrame from the events tree of the input file.
rdf = ROOT.RDataFrame("output", "../Data/run7722all_xyz.root")
# Extract nhits values
digitNhits = rdf.AsNumpy(columns=["digitNhits"])["digitNhits"]
# Histogramming
digitNhits_counts, digitNhits_bins = np.histogram(digitNhits, bins=np.arange(0, 240, 1))
digitNhits_bin_centers = [(digitNhits_bins[i] + digitNhits_bins[i + 1]) / 2 for i in range(len(digitNhits_bins) - 1)]
# Plotting
plt.figure(figsize=(12, 6))
plt.title(f"nhits histogram")
plt.errorbar(digitNhits_bin_centers, digitNhits_counts,
             ls='', marker='o', mfc='black', ms=4, mec='black',
             ecolor='black', label="Run 7722")

plt.xlabel("digitNhits", fontsize=18)
plt.ylabel("NEvents", fontsize=15)
plt.yscale('log')
#plt.ylim(3000,250_000)
plt.legend()
plt.show()

This digitNhtis looks pretty normal for Muon Data. There's probably a lot of different events and noise mix into it but hopefully the Muon paddle data will help with this.

In [ ]:
# This prints out the meta tree.
import ROOT
f = ROOT.TFile.Open("../Data/run7722all_xyz.root")
meta = f.Get("meta")
meta.Print()

We need the values from muon_paddle_x,y,z.

In [1]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt

meta = ROOT.RDataFrame("meta", "../Data/run7722all_xyz.root")

#muon_paddle_x = meta.AsNumpy(columns=["muon_paddle_x"])
#print(muon_paddle_x)

#muon_paddle_y = meta.AsNumpy(columns=["muon_paddle_y"])
#print(muon_paddle_y)

muon_paddle_z = meta.AsNumpy(columns=["muon_paddle_z"])["muon_paddle_z"][0]
print(muon_paddle_z.shape)
#print(muon_paddle_z[0].shape)

(134,)


muon_paddle_x,y,z seems to be 164x134 shape. The 134 is for the number of paddles, 164 is actually from the number of root/hdf5 files that were combined together.

What I want to do is to combine the position in the meta and hits from output together into a single array for analysis.

In [2]:
# Code to combine layer#, xyz positions of the paddle and the event by event hits.
# Layer is the row 0, xyz are row 1 to 3, hit data are 4 to NEvents+4
import ROOT
import numpy as np
import matplotlib.pyplot as plt

meta = ROOT.RDataFrame("meta", "../Data/run7722all_xyz.root")
# Get the xyz position, but we just need the 1st entry (hdf5 file).
x_pos = meta.AsNumpy(columns=["muon_paddle_x"])["muon_paddle_x"][0]
y_pos = meta.AsNumpy(columns=["muon_paddle_y"])["muon_paddle_y"][0]
z_pos = meta.AsNumpy(columns=["muon_paddle_z"])["muon_paddle_z"][0]
#print("x_pos shape:", x_pos.shape)
#print("y_pos shape:", y_pos.shape)
#print("z_pos shape:", z_pos.shape)

output = ROOT.RDataFrame("output", "../Data/run7722all_xyz.root")
# Get the muon_paddle_hit from the output tree.
# Convert from object array of event vectors to 2D array: NEvents x 134
muon_hits = np.stack(output.AsNumpy(["muon_paddle_hit"])["muon_paddle_hit"])

print("muon_hits shape:", muon_hits.shape)

# Put a value for the layers 
layer = np.zeros_like(z_pos, dtype=int) # defaults to 0
layer[np.isclose(z_pos, 2129.984)] = 1 # TopUpper-x
layer[np.isclose(z_pos, 2098.311)] = 2 # TopUpper+x
layer[np.isclose(z_pos, 1972.377)] = 3 # TopLower-y
layer[np.isclose(z_pos, 1940.704)] = 4 # TopLower+y
layer[np.isclose(z_pos, 699.896)] = 5 # Barrel+z
layer[np.isclose(z_pos, -398.908)] = 6 # Barrel-z
layer[np.isclose(z_pos, -1807.338)] = 7 # Bottom-y
layer[np.isclose(z_pos, -1839.012)] = 8 # Bottom+y

# Stack x, y, z as first 3 rows
muon_pos_hits = np.vstack([
    layer,
    x_pos,
    y_pos,
    z_pos,
    muon_hits
])

print("muon_hits_with_pos shape:", muon_pos_hits[0])

muon_hits shape: (353719, 134)
muon_hits_with_pos shape: [6. 5. 6. 3. 6. 5. 3. 4. 6. 5. 0. 0. 8. 2. 2. 3. 4. 8. 8. 8. 8. 8. 6. 6.
 6. 6. 6. 8. 8. 8. 8. 8. 6. 6. 6. 6. 6. 6. 6. 6. 3. 6. 3. 3. 3. 4. 4. 7.
 1. 1. 4. 0. 7. 7. 7. 7. 7. 5. 5. 5. 5. 5. 7. 7. 7. 7. 7. 5. 5. 5. 5. 5.
 5. 5. 5. 4. 5. 4. 0. 3. 4. 0. 6. 6. 5. 5. 6. 6. 5. 5. 6. 6. 5. 5. 6. 6.
 5. 5. 6. 6. 5. 5. 6. 6. 5. 5. 0. 0. 2. 6. 1. 5. 2. 6. 1. 5. 2. 2. 1. 1.
 2. 2. 1. 1. 2. 2. 1. 1. 3. 6. 4. 5. 0. 0.]


In [2]:
# Just checking the xyz for particular z position paddles
print(muon_pos_hits[1:4,np.where(np.isclose(muon_pos_hits[3,], -1839.012))[0]])

[[ 1270.     1016.      762.      508.      254.        0.    -1270.
  -1016.     -762.     -508.     -254.   ]
 [  549.402   549.402   549.402   549.402   549.402   549.402   549.402
    549.402   549.402   549.402   549.402]
 [-1839.012 -1839.012 -1839.012 -1839.012 -1839.012 -1839.012 -1839.012
  -1839.012 -1839.012 -1839.012 -1839.012]]


## Let's look for muon decay events.
The muon decays TopUpper, TopLower with 1 hit and no hits on the bottom or barrel.<br>

Let's filter for the events that has 1 hit on each TopUpper, TopLower and no hits on the bottom and barrel.

In [11]:
import numpy as np


def xy_bounds(center, size):
    """Return projected x-y bounds: xmin, xmax, ymin, ymax."""
    x, y, _ = center
    dx, dy, _ = size

    return (
        x - dx / 2.0,
        x + dx / 2.0,
        y - dy / 2.0,
        y + dy / 2.0,
    )


def projected_xy_overlap(center_a, size_a, center_b, size_b):
    """
    Check whether two boxes have positive-area overlap when viewed from +z.

    Returns
    -------
    overlaps : bool
    overlap_x : float
    overlap_y : float
    overlap_area : float
    """
    ax0, ax1, ay0, ay1 = xy_bounds(center_a, size_a)
    bx0, bx1, by0, by1 = xy_bounds(center_b, size_b)

    overlap_x = min(ax1, bx1) - max(ax0, bx0)
    overlap_y = min(ay1, by1) - max(ay0, by0)

    overlaps = overlap_x > 0 and overlap_y > 0
    overlap_area = overlap_x * overlap_y if overlaps else 0.0

    return overlaps, max(overlap_x, 0.0), max(overlap_y, 0.0), overlap_area


def find_top_layer_overlap_events(muon_pos_hits):
    """
    Find events having:
      - exactly one hit across layers 1 and 2
      - exactly one hit across layers 3 and 4
      - overlapping x-y projections for those two hits

    Assumes:
      row 0: layer number
      row 1: x position
      row 2: y position
      row 3: z position
      rows 4 onward: event hit masks
    """
    layers = muon_pos_hits[0].astype(int)

    upper_columns = np.where(np.isin(layers, [1, 2]))[0]
    lower_columns = np.where(np.isin(layers, [3, 4]))[0]

    matching_events = []

    number_of_events = muon_pos_hits.shape[0] - 4

    for event_num in range(number_of_events):
        event_row = event_num + 4
        event_hits = muon_pos_hits[event_row] == 1

        upper_hit_columns = upper_columns[event_hits[upper_columns]]
        lower_hit_columns = lower_columns[event_hits[lower_columns]]

        # Require exactly one total hit in each layer group
        if len(upper_hit_columns) != 1:
            continue

        if len(lower_hit_columns) != 1:
            continue

        upper_col = upper_hit_columns[0]
        lower_col = lower_hit_columns[0]

        upper_layer = int(layers[upper_col])
        lower_layer = int(layers[lower_col])

        upper_center = np.array([
            muon_pos_hits[1, upper_col],
            muon_pos_hits[2, upper_col],
            muon_pos_hits[3, upper_col],
        ], dtype=float)

        lower_center = np.array([
            muon_pos_hits[1, lower_col],
            muon_pos_hits[2, lower_col],
            muon_pos_hits[3, lower_col],
        ], dtype=float)

        upper_size = box_size_from_layer(upper_layer)
        lower_size = box_size_from_layer(lower_layer)

        overlaps, overlap_x, overlap_y, overlap_area = projected_xy_overlap(
            upper_center,
            upper_size,
            lower_center,
            lower_size,
        )

        if overlaps:
            matching_events.append({
                "event": event_num,
                "upper_col": int(upper_col),
                "upper_layer": upper_layer,
                "upper_center": upper_center,
                "lower_col": int(lower_col),
                "lower_layer": lower_layer,
                "lower_center": lower_center,
                "overlap_x": overlap_x,
                "overlap_y": overlap_y,
                "overlap_area": overlap_area,
            })

    return matching_events

In [12]:
matching_events = find_top_layer_overlap_events(muon_pos_hits)

print(f"Number of matching events: {len(matching_events)}")

for match in matching_events[:20]:
    print(
        f"Event {match['event']}: "
        f"upper layer {match['upper_layer']} col {match['upper_col']}, "
        f"lower layer {match['lower_layer']} col {match['lower_col']}, "
        f"overlap = {match['overlap_x']:.1f} x "
        f"{match['overlap_y']:.1f}, "
        f"area = {match['overlap_area']:.1f}"
    )

Number of matching events: 645
Event 102: upper layer 2 col 117, lower layer 4 col 46, overlap = 210.0 x 210.0, area = 44100.0
Event 407: upper layer 1 col 118, lower layer 3 col 3, overlap = 210.0 x 210.0, area = 44100.0
Event 677: upper layer 1 col 126, lower layer 3 col 3, overlap = 210.0 x 210.0, area = 44100.0
Event 859: upper layer 1 col 126, lower layer 3 col 3, overlap = 210.0 x 210.0, area = 44100.0
Event 921: upper layer 2 col 121, lower layer 4 col 46, overlap = 210.0 x 210.0, area = 44100.0
Event 1065: upper layer 1 col 122, lower layer 3 col 3, overlap = 210.0 x 210.0, area = 44100.0
Event 1456: upper layer 1 col 126, lower layer 3 col 3, overlap = 210.0 x 210.0, area = 44100.0
Event 1775: upper layer 1 col 126, lower layer 3 col 3, overlap = 210.0 x 210.0, area = 44100.0
Event 1792: upper layer 1 col 118, lower layer 3 col 3, overlap = 210.0 x 210.0, area = 44100.0
Event 1915: upper layer 1 col 122, lower layer 3 col 3, overlap = 210.0 x 210.0, area = 44100.0
Event 2835: 

In [15]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
# Use Qt backend for smoother 3D rotation
# Run this once in a separate notebook cell if needed:
%matplotlib qt

EVENTNUM = 407

def draw_box(ax, center, size, alpha=0.35):
    """
    Draw a rectangular box centered at center=(x,y,z)
    with full dimensions size=(dx,dy,dz).
    """
    cx, cy, cz = center
    dx, dy, dz = size

    x0, x1 = cx - dx/2, cx + dx/2
    y0, y1 = cy - dy/2, cy + dy/2
    z0, z1 = cz - dz/2, cz + dz/2

    vertices = np.array([
        [x0, y0, z0],
        [x1, y0, z0],
        [x1, y1, z0],
        [x0, y1, z0],
        [x0, y0, z1],
        [x1, y0, z1],
        [x1, y1, z1],
        [x0, y1, z1],
    ])

    faces = [
        [vertices[0], vertices[1], vertices[2], vertices[3]],  # bottom
        [vertices[4], vertices[5], vertices[6], vertices[7]],  # top
        [vertices[0], vertices[1], vertices[5], vertices[4]],  # side
        [vertices[2], vertices[3], vertices[7], vertices[6]],  # side
        [vertices[1], vertices[2], vertices[6], vertices[5]],  # side
        [vertices[0], vertices[3], vertices[7], vertices[4]],  # side
    ]

    box = Poly3DCollection(
        faces,
        alpha=alpha,
        edgecolor="k",
        linewidths=0.8
    )

    ax.add_collection3d(box)

event_hit_row = 4 + EVENTNUM

hit_columns = np.where(muon_pos_hits[event_hit_row] == 1)[0]

print(f"Event {EVENTNUM}")
print("hit columns:", hit_columns)
print("number of hits:", len(hit_columns))

fig = plt.figure(figsize=(10, 11))
ax = fig.add_subplot(111, projection="3d")

for col in hit_columns:
    layer_value = int(muon_pos_hits[0, col])

    x = muon_pos_hits[1, col]
    y = muon_pos_hits[2, col]
    z = muon_pos_hits[3, col]

    print(f"  col {col}: layer={layer_value}, x={x}, y={y}, z={z}")

    if layer_value == 1:
        draw_box(ax, center=(x, y, z), size=(1300, 210, 10), alpha=0.35)
    elif layer_value == 2:
        draw_box(ax, center=(x, y, z), size=(1300, 210, 10), alpha=0.35)
    elif layer_value == 3:
        draw_box(ax, center=(x, y, z), size=(210, 1300, 10), alpha=0.35)    
    elif layer_value == 4:
        draw_box(ax, center=(x, y, z), size=(210, 1300, 10), alpha=0.35)
    elif layer_value == 5:
        draw_box(ax, center=(x, y, z), size=(210, 210, 1300), alpha=0.35)
    elif layer_value == 6:
        draw_box(ax, center=(x, y, z), size=(210, 210, 1300), alpha=0.35)
    elif layer_value == 7:
        draw_box(ax, center=(x, y, z), size=(210, 1300, 10), alpha=0.35)    
    elif layer_value == 8:
        draw_box(ax, center=(x, y, z), size=(210, 1300, 10), alpha=0.35)
    else:
        ax.scatter(x, y, z, s=80)

    ax.text(x, y, z, layer_value, fontsize=9)


ax.set_xlabel("x_pos")
ax.set_ylabel("y_pos")
ax.set_zlabel("z_pos")
ax.set_title(f"Event {EVENTNUM}: Muon Hits with Layer #")

ax.set_xlim(-1600, 1600)
ax.set_ylim(-1600, 1600)
ax.set_zlim(-2200, 2200)

ax.set_box_aspect((3200, 3200, 4400))

plt.show()

Event 407
hit columns: [  3 118]
number of hits: 2
  col 3: layer=3, x=-704.85, y=-549.402, z=1972.377
  col 118: layer=1, x=-549.402, y=-704.85, z=2129.984


In [17]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt

rdf = ROOT.RDataFrame(
    "output",
    "../Data/run7722all_xyz.root"
)

digitNhits = rdf.AsNumpy(columns=["digitNhits"])["digitNhits"]
n_events = len(digitNhits)

# Extract the matching event indices
event_match = np.array(
    sorted({match["event"] for match in matching_events}),
    dtype=int
)

# Keep only valid matching-event indices
event_match = event_match[
    (event_match >= 0) & (event_match < n_events)
]

# Select the event immediately after each matching event.
# A matching event at n_events - 1 has no following event.
event_after_match = event_match[event_match + 1 < n_events] + 1

# Remove duplicates in case consecutive matching events produce
# the same selected event through some upstream duplication.
event_after_match = np.unique(event_after_match)

# digitNhits for the events after matching events
digitNhits_after_match = digitNhits[event_after_match]

# digitNhits for every event in the ROOT file
digitNhits_all = digitNhits

bins = np.arange(0, 240, 5)

after_counts, digitNhits_bins = np.histogram(
    digitNhits_after_match,
    bins=bins
)

all_counts, _ = np.histogram(
    digitNhits_all,
    bins=bins
)

# Scale the all-events histogram to the same total as the
# after-match histogram so their distribution shapes can be compared.
after_total = after_counts.sum()
all_total = all_counts.sum()

all_counts_scaled = all_counts.astype(float)

if all_total > 0:
    all_counts_scaled *= after_total / all_total

digitNhits_bin_centers = 0.5 * (
    digitNhits_bins[:-1] + digitNhits_bins[1:]
)

plt.figure(figsize=(12, 6))

plt.title(
    "digitNhits: events after matched events vs all events"
)

plt.plot(
    digitNhits_bin_centers,
    after_counts,
    color="blue",
    label=(
        "Events after matched events "
        f"({len(event_after_match)} events)"
    )
)

plt.plot(
    digitNhits_bin_centers,
    all_counts_scaled,
    color="red",
    label=(
        f"All events ({n_events} events, normalized)"
    )
)

plt.xlabel("digitNhits", fontsize=18)
plt.ylabel("NEvents", fontsize=15)
plt.yscale("log")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Matching events: {len(event_match)}")
print(f"Events after matching events: {len(event_after_match)}")
print(f"All events: {n_events}")

Matching events: 645
Events after matching events: 645
All events: 353719


In [24]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt

rdf = ROOT.RDataFrame(
    "output",
    "../Data/run7722all_xyz.root"
)

time_since_last_trigger = rdf.AsNumpy(
    columns=["timeSinceLastTrigger_us"]
)["timeSinceLastTrigger_us"]

n_events = len(time_since_last_trigger)

# Extract matching event indices
event_match = np.array(
    sorted({match["event"] for match in matching_events}),
    dtype=int
)

# Keep only valid matching-event indices
event_match = event_match[
    (event_match >= 0) & (event_match < n_events)
]

# Select the event immediately after each matching event
event_after_match = event_match[event_match + 1 < n_events] + 1
event_after_match = np.unique(event_after_match)

# Values for events immediately after matched events
time_after_match = time_since_last_trigger[event_after_match]

# Values for all events
time_all = time_since_last_trigger

# Choose histogram bins from the data range.
# Percentile clipping prevents a few extreme values from stretching the plot.
finite_values = time_all[np.isfinite(time_all)]

if len(finite_values) == 0:
    raise ValueError("No finite timeSinceLastTrigger_us values found.")

x_max = np.percentile(finite_values, 99.5)

bins = np.linspace(
    0,
    20,
    100
)

after_counts, time_bins = np.histogram(
    time_after_match,
    bins=bins
)

all_counts, _ = np.histogram(
    time_all,
    bins=bins
)

# Normalize all-events histogram to the selected-event histogram
after_total = after_counts.sum()
all_total = all_counts.sum()

all_counts_scaled = all_counts.astype(float)

if all_total > 0:
    all_counts_scaled *= after_total / all_total

time_bin_centers = 0.5 * (
    time_bins[:-1] + time_bins[1:]
)

plt.figure(figsize=(12, 6))

plt.title(
    "Time since last trigger: events after matched events vs all events"
)

plt.plot(
    time_bin_centers,
    after_counts,
    color="blue",
    label=(
        "Events after matched events "
        f"({len(event_after_match)} events)"
    )
)

plt.plot(
    time_bin_centers,
    all_counts_scaled,
    color="red",
    label=(
        f"All events ({n_events} events, normalized)"
    )
)

plt.xlabel("timeSinceLastTrigger_us [µs]", fontsize=18)
plt.ylabel("NEvents", fontsize=15)
#plt.yscale("log")
plt.legend()
plt.tight_layout()
#plt.ylim(1e-5,40)
plt.show()

print(f"Matching events: {len(event_match)}")
print(f"Events after matching events: {len(event_after_match)}")
print(f"All events: {n_events}")
print(f"Histogram range: 0 to {x_max:.3f} µs")

Matching events: 645
Events after matching events: 645
All events: 353719
Histogram range: 0 to 63577.865 µs
